<a href="https://colab.research.google.com/github/abdelruhman161-cyber/Assignment-1/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdelruhman161-cyber/Assignment-1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [4]:
import os
import json
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from sklearn.ensemble import RandomForestClassifier

# 1. Connection and DuckDB HTTPFS Setup
try:
    hf_token_val = userdata.get('HF_TOKEN')
except Exception:
    hf_token_val = os.environ.get('HF_TOKEN')

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE HUGGINGFACE, TOKEN '{hf_token_val}');")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT_PERFORMANCE = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# Detect correct date column dynamically
schema_df = con.execute(f"DESCRIBE SELECT * FROM {DIM_CONTENT}").df()
cols = [c.lower() for c in schema_df['column_name'].tolist()]
date_col = 'first_seen_date' if 'first_seen_date' in cols else ('published_at' if 'published_at' in cols else ('content_created_date' if 'content_created_date' in cols else 'created_at'))

# 2. Extract Data (March 2026 Features vs April 2026 Ground Truth)
query_pb = f"""
WITH m3 AS (
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        COALESCE(DATEDIFF('day', TRY_CAST(c.{date_col} AS DATE), DATE '2026-03-31'), 0) as page_age_days,
        SUM(f.gsc_impressions) as imp_m3,
        SUM(f.gsc_clicks) as clicks_m3,
        AVG(f.gsc_sum_position) as pos_m3,
        CASE WHEN SUM(f.gsc_impressions) > 0 THEN (SUM(f.gsc_clicks)::FLOAT / SUM(f.gsc_impressions)) ELSE 0 END as ctr_m3
    FROM {FACT_PERFORMANCE} f
    LEFT JOIN {DIM_CONTENT} c ON f.content_hash_id = c.content_hash_id
    WHERE STRFTIME(f.report_date, '%Y-%m') = '2026-03'
    GROUP BY f.content_hash_id, f.client_hash_id, c.{date_col}
),
m4 AS (
    SELECT
        content_hash_id,
        SUM(gsc_impressions) as imp_m4
    FROM {FACT_PERFORMANCE}
    WHERE STRFTIME(report_date, '%Y-%m') = '2026-04'
    GROUP BY content_hash_id
)
SELECT
    m3.*,
    CASE
        WHEN m4.imp_m4 IS NULL OR m3.imp_m3 = 0 THEN 0
        WHEN ((m4.imp_m4 - m3.imp_m3)::FLOAT / m3.imp_m3) < -0.20 THEN 1
        ELSE 0
    END as is_declining
FROM m3
LEFT JOIN m4 ON m3.content_hash_id = m4.content_hash_id
WHERE m3.imp_m3 >= 100;
"""

df_pb = con.execute(query_pb).df().fillna(0)

# 3. Fit Random Forest & Predict Decay Probabilities
feature_cols = ['imp_m3', 'clicks_m3', 'pos_m3', 'ctr_m3', 'page_age_days']
X = df_pb[feature_cols]
y = df_pb['is_declining']

rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf.fit(X, y)
df_pb['decay_probability'] = rf.predict_proba(X)[:, 1]

# 4. Map Archetype, Action Label, and Review Priority
def assign_action(row):
    if row['decay_probability'] >= 0.60 and row['page_age_days'] > 365:
        return 'DECAY_STALE_HIGH_TRAFFIC', 'REFRESH_CONTENT', 'HIGH'
    elif row['pos_m3'] <= 10 and row['ctr_m3'] < 0.01:
        return 'CTR_UNDERPERFORMER', 'OPTIMIZE_METATAGS', 'MEDIUM'
    else:
        return 'STABLE_PERFORMER', 'MONITOR', 'LOW'

df_pb[['reason_code', 'action_label', 'review_priority']] = df_pb.apply(
    assign_action, axis=1, result_type='expand'
)

# Sort queue descending by decay probability
playbook_queue = df_pb.sort_values(by='decay_probability', ascending=False).reset_index(drop=True)
print(f"✅ Playbook Queue Generated: {len(playbook_queue):,} pages scored.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Playbook Queue Generated: 101,441 pages scored.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [5]:
* **Decision-Support Tool:** Designed to assist editorial teams in triaging large content catalogs for quarterly audit cycles.
* **Non-Autonomous:** The playbook outputs prioritize recommendations; it does NOT automatically publish or rewrite content.

1. **Seasonal Queries:** The model may flag high-traffic seasonal pages experiencing expected off-peak drops as decaying.
2. **Technical SERP Changes:** Algorithmic layout shifts (e.g., AI Overviews) impacting CTR are not separated from underlying content quality decay.

SyntaxError: invalid syntax (3286194421.py, line 1)

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
Before executing any `REFRESH_CONTENT` action, an editor must verify:
* [ ] The keyword intent has not fundamentally shifted.
* [ ] The page was not recently migrated or URL-redirected within the last 90 days.
* [ ] Traffic loss is not tied to a temporary product out-of-stock event.

1. **Automated Publishing / Content Overwrites:** Never auto-replace live URL content without human copy-editing.
2. **Deleting or Unindexing Pages:** Pruning content must undergo legal and brand compliance checks.
3. **Core Landing / Conversion Pages:** High-value transactional pages must be reviewed manually regardless of decay score.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
### Monitoring & Retrain Triggers
* **Performance Drift Trigger:** Retrain the decay model if the validation ROC-AUC drops below **0.65** on rolling 30-day windows.
* **Schema Drift Trigger:** Re-audit features if missing values in `gsc_impressions` exceed **5%** across active clients.
* **Cost / Value Assessment:** Prioritize human review on pages generating > 1,000 monthly impressions to maximize ROI on editorial hours.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [6]:
# 1. Export Ranked Queue CSV to work/outputs/
os.makedirs('../outputs', exist_ok=True)
export_cols = ['content_hash_id', 'decay_probability', 'reason_code', 'action_label', 'review_priority', 'imp_m3', 'page_age_days']
playbook_queue[export_cols].to_csv('../outputs/w07_playbook_queue.csv', index=False)

# 2. Export Metrics Receipt JSON to work/outputs/
metrics_receipt = {
    "total_scored_pages": int(len(playbook_queue)),
    "refresh_content_count": int((playbook_queue['action_label'] == 'REFRESH_CONTENT').sum()),
    "optimize_metatags_count": int((playbook_queue['action_label'] == 'OPTIMIZE_METATAGS').sum()),
    "monitor_count": int((playbook_queue['action_label'] == 'MONITOR').sum()),
    "mean_decay_probability": float(playbook_queue['decay_probability'].mean())
}

with open('../outputs/w07_playbook_metrics.json', 'w') as f:
    json.dump(metrics_receipt, f, indent=4)

print("✅ Saved CSV queue to work/outputs/w07_playbook_queue.csv")
print("✅ Saved metrics receipt to work/outputs/w07_playbook_metrics.json")

✅ Saved CSV queue to work/outputs/w07_playbook_queue.csv
✅ Saved metrics receipt to work/outputs/w07_playbook_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.